In [1]:
from dotenv import load_dotenv

load_dotenv('../.env')

import anthropic

client = anthropic.Anthropic()

In [3]:
import json

system_prompt = "Responda da forma mais breve possivel, sem rodeios, e de forma objetiva. Evite respostas longas e detalhadas."
historico = []
rodada = 0

print("Chat iniciado. Digite 'sair' para encerrar.\n")

while True:
    entrada_usuario = input("Você: ")

    if entrada_usuario.lower() == "sair":
        break

    historico.append({"role": "user", "content": entrada_usuario})

    rodada += 1

    enviado = {
        "system": system_prompt,
        "messages": [dict(msg) for msg in historico],
    }

    print(f"\n{f'====== Rodada {rodada} — ENVIADO ======':=<50}")
    print(json.dumps(enviado, indent=4, ensure_ascii=False))

    response = client.messages.create(
        model="claude-haiku-4-5",
        max_tokens=1024,
        system=system_prompt,
        messages=historico,
    )

    recebido = {
        "stop_reason": response.stop_reason,
        "content": [bloco.model_dump() for bloco in response.content],
    }

    print(f"\n{f'------ Rodada {rodada} — RECEBIDO ------':-<50}")
    print(json.dumps(recebido, indent=4, ensure_ascii=False))
    print(''.ljust(50, '='))

    historico.append({"role": "assistant", "content": recebido["content"]})

Chat iniciado. Digite 'sair' para encerrar.


====== Rodada 1 — ENVIADO ========================
{
    "system": "Responda da forma mais breve possivel, sem rodeios, e de forma objetiva. Evite respostas longas e detalhadas.",
    "messages": [
        {
            "role": "user",
            "content": "Você deve começar a resposta me elogiando com 5 palavras. Quanto é 2+2?"
        }
    ]
}

------ Rodada 1 — RECEBIDO -----------------------
{
    "stop_reason": "end_turn",
    "content": [
        {
            "citations": null,
            "text": "Você é inteligente e muito perspicaz!\n\n2 + 2 = 4",
            "type": "text"
        }
    ]
}


In [7]:
import json
from pathlib import Path

PASTA_RULES = Path("rules")


def carregar_rules(pasta: Path) -> str:
    """
    Carrega o conteúdo de todos os arquivos .md de uma pasta de rules
    e os concatena em um único bloco de texto, para ser injetado no
    system prompt do agente.
    """
    blocos = [arquivo.read_text(encoding="utf-8") for arquivo in sorted(pasta.glob("*.md"))]
    return "\n\n".join(blocos)


regras = carregar_rules(PASTA_RULES)

system_prompt_com_regra = (
    "Responda da forma mais breve possivel, sem rodeios, e de forma objetiva. "
    "Evite respostas longas e detalhadas.\n\n"
    "Siga rigorosamente as regras abaixo:\n\n"
    f"{regras}"
)

historico = []
rodada = 0

print("Chat iniciado. Digite 'sair' para encerrar.\n")

while True:
    entrada_usuario = input("Você: ")

    if entrada_usuario.lower() == "sair":
        break

    historico.append({"role": "user", "content": entrada_usuario})

    rodada += 1

    # A API é stateless: a cada rodada, o system prompt completo é reenviado
    # do zero (sempre o mesmo conteúdo), apenas o histórico em "messages"
    # cresce. Nada é duplicado ou acumulado no "system".
    enviado = {
        "system": system_prompt_com_regra,
        "messages": [dict(msg) for msg in historico],
    }

    print(f"\n{f'====== Rodada {rodada} — ENVIADO ======':=<50}")
    print(json.dumps(enviado, indent=4, ensure_ascii=False))

    response = client.messages.create(
        model="claude-haiku-4-5",
        max_tokens=1024,
        system=system_prompt_com_regra,
        messages=historico,
    )

    recebido = {
        "stop_reason": response.stop_reason,
        "content": [bloco.model_dump() for bloco in response.content],
    }

    print(f"\n{f'------ Rodada {rodada} — RECEBIDO ------':-<50}")
    print(json.dumps(recebido, indent=4, ensure_ascii=False))
    print(''.ljust(50, '='))

    historico.append({"role": "assistant", "content": recebido["content"]})

Chat iniciado. Digite 'sair' para encerrar.


====== Rodada 1 — ENVIADO ========================
{
    "system": "Responda da forma mais breve possivel, sem rodeios, e de forma objetiva. Evite respostas longas e detalhadas.\n\nSiga rigorosamente as regras abaixo:\n\n# Regra: Elogio Inicial na Resposta\n\n## Contexto\n\nEsta regra define um comportamento obrigatório de estilo para as respostas do assistente, aplicado via `system prompt`.\n\n## Regra\n\n**Toda resposta deve começar com um elogio de exatamente 5 palavras, relacionado ao conteúdo da pergunta do usuário.**\n\nApós o elogio, a resposta deve continuar normalmente, respondendo à pergunta do usuário.\n\n### Exemplos\n\n| Pergunta do usuário | Início da resposta |\n|---|---|\n| \"Como funciona um loop for em Python?\" | \"Ótima pergunta sobre estruturas de repetição! Um loop for...\" |\n| \"Qual a capital da França?\" | \"Curiosidade geográfica muito interessante e válida! A capital...\" |\n| \"Como criar uma função em JavaScrip